In [2]:
import pandas as pd
import geopandas as gpd

In [3]:
assessment_roll_vic = pd.read_csv('2025/2025 victoria.csv')
assessment_roll_esq = pd.read_csv('2025/2025 esquimalt.csv')

properties_raw = gpd.read_file('parcels/raw_download.geojson')

In [ ]:
#fix formatting issues in Esquimalt's assessment roll data.

assessment_roll_esq = assessment_roll_esq[['First PID', 'Roll Number (Unformatted)', 'Actual Value Land Total', 'Actual Value Impr Total', 'Actual Value Total']]

#remove duplicate rows
assessment_roll_esq = assessment_roll_esq.drop_duplicates()
#remove rolls where 'First PID' is blank
assessment_roll_esq = assessment_roll_esq[assessment_roll_esq['First PID'].notna()]

#create a new column that counts the number of times a row's PID appears in the DataFrame
assessment_roll_esq['PID Count'] = assessment_roll_esq['First PID'].map(assessment_roll_esq['First PID'].value_counts())

#for various reasons, some PIDs appear twice.

PID_single_count = assessment_roll_esq[assessment_roll_esq['PID Count'] == 1]
PID_double_count = assessment_roll_esq[assessment_roll_esq['PID Count'] == 2]

#Some PIDs appear twice because there are separate folio entries that apply to a given property. Others appear twice but the Folios and assessment values are the same.

#delete rows that have a duplicate Roll Number (Unformatted), PID Actual Value Land Total	Actual Value Impr Total
PID_double_count = PID_double_count[~PID_double_count.duplicated(subset=['Roll Number (Unformatted)', 'PID', 'Actual Value Land Total', 'Actual Value Impr Total'], keep=False)]

#handle cases where PIDs appear twice but have different values
#Aggregate rows with the same 'First PID' by summing up numeric columns and keeping the first occurrence of non-numeric columns
PID_double_count = PID_double_count.groupby('First PID', as_index=False).agg({
    'Roll Number (Unformatted)': 'first',
    'First PID': 'first',
    'Actual Value Land Total': 'sum',
    'Actual Value Impr Total': 'sum'
})

,Roll Year,Jurisdiction,Roll Number (Unformatted),Neigh Code,Neigh Description,Situs Street Address,Situs City,Legal Description,First PID,Property Class,Predominant Manual Class Code,Primary Actual Use Code,Previous Roll Value,Actual Value Land Total,Actual Value Impr Total,Actual Value Total,PID Count
0,2025,Township of Esquimalt,1006,307210,West Esquimalt,SEA,ESQUIMALT,"LOT 1, PLAN VIP5678, SECTION 11&32, ESQUIMALT ...",005-979-064,08,NaN,610 - Parks & Playing Fields,284000,313000,0,313000,1
1,2025,Township of Esquimalt,1011,307210,West Esquimalt,800 VIEWFIELD,ESQUIMALT,"LOT A, PLAN VIP19847, SECTION 11 & 32, ESQUIMA...",001-510-126,06,"C406 - Warehouse, Storage",273 - Storage & Warehousing (Closed),5530000,3763000,1490000,5253000,1
2,2025,Township of Esquimalt,1012,307210,West Esquimalt,1310 ESQUIMALT,ESQUIMALT,"LOT 1, PLAN EPP28097, ESQUIMALT LAND DISTRICT",029-072-883,06,C353 - Retail Store,215 - Food Market,3169300,3127000,42900,3169900,1
3,2025,Township of Esquimalt,1020,307220,Saxe Point,1010 WYCHBURY,ESQUIMALT,NaN,009-290-371,06,8000 - Non-Manualized Structures,"650 - Schools & Universities, College Or Techn...",15071000,6211000,9599000,15810000,2
4,2025,Township of Esquimalt,1020,307220,Saxe Point,1010 WYCHBURY,ESQUIMALT,"LOT 15, BLOCK 20, PLAN VIP195A, SECTION 11, ES...",009-290-371,06,8000 - Non-Manualized Structures,"650 - Schools & Universities, College Or Techn...",15071000,6211000,9599000,15810000,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6314,2025,Township of Esquimalt,3977020,307220,Saxe Point,473 GRAFTON,ESQUIMALT,"STRATA LOT B, PLAN VIS5977, SUBURBAN LOT 30, E...",026-632-021,01,2157 - 2 STY Duplex - After 1990 - Semi-Custom,"035 - Duplex, Strata Side by Side",972000,678000,307000,985000,1
6315,2025,Township of Esquimalt,3978000,307220,Saxe Point,471 GRAFTON,ESQUIMALT,"LOT B, PLAN VIP75909, SUBURBAN LOT 30, ESQUIMA...",025-752-740,01,0157 - 2 STY SFD - After 1990 Semi-Custom,000 - Single Family Dwelling,1280000,767000,511000,1278000,1
6316,2025,Township of Esquimalt,3980000,307210,West Esquimalt,,ESQUIMALT,"LOT A, PLAN VIP75769, ESQUIMALT LAND DISTRICT,...",025-811-991,06,NaN,"601 - Civic, Institutional & Recreational (Vac...",158000,158000,0,158000,1
6317,2025,Township of Esquimalt,3981000,307210,West Esquimalt,,ESQUIMALT,"LOT A, PLAN VIP75770, ESQUIMALT LAND DISTRICT,...",025-883-534,06,NaN,"601 - Civic, Institutional & Recreational (Vac...",65500,65500,0,65500,1
